In [1]:
# ================================================================
# 🎓 LECTUREMIND — COMPLETE STREAMLIT PROJECT
# ONE GOOGLE COLAB CELL
#
# EXACTLY 4 CATEGORIES:
# 1. 📚 Lecture
# 2. 🤖 AI Tutor
# 3. 📝 Study Guide
# 4. 🧪 Quiz Generator
#
# LLM: OpenRouter
# ================================================================


# ================================================================
# 1. INSTALL DEPENDENCIES
# ================================================================

import subprocess
import sys
import os
import time

print("📦 Installing dependencies...")

subprocess.run(
    "apt-get -qq update && apt-get -qq install -y ffmpeg",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "faster-whisper",
        "sentence-transformers",
        "faiss-cpu",
        "openai",
        "streamlit"
    ],
    check=True
)

print("✅ Dependencies installed")


# ================================================================
# 2. CREATE STREAMLIT APP
# ================================================================

app_code = r'''
import streamlit as st
import os
import json
import hashlib
import subprocess
import numpy as np
import torch
import time

from pathlib import Path

import faiss

from faster_whisper import WhisperModel

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from openai import OpenAI


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="LectureMind",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="collapsed"
)


# ============================================================
# CSS
# ============================================================

st.markdown(
    """
    <style>

    .main-title {
        font-size: 3.0rem;
        font-weight: 800;
        margin-bottom: 0;
    }

    .subtitle {
        font-size: 1.25rem;
        font-weight: 600;
        color: #555;
    }

    .source-box {
        padding: 16px;
        border-radius: 12px;
        border: 1px solid #ddd;
        margin: 10px 0;
        background: #fafafa;
    }

    .success-box {
        padding: 15px;
        border-radius: 12px;
        border: 1px solid #b7dfb9;
        background: #f1fff2;
    }

    </style>
    """,
    unsafe_allow_html=True
)


# ============================================================
# DIRECTORIES
# ============================================================

BASE_DIR = Path("/content/lecturemind")

VIDEO_DIR = BASE_DIR / "videos"
AUDIO_DIR = BASE_DIR / "audio"
DATA_DIR = BASE_DIR / "data"

VIDEO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

AUDIO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CONFIGURATION
# ============================================================

WHISPER_MODEL = "small"

EMBEDDING_MODEL = (
    "sentence-transformers/"
    "all-MiniLM-L6-v2"
)

RERANKER_MODEL = (
    "cross-encoder/"
    "ms-marco-MiniLM-L-6-v2"
)

INITIAL_TOP_K = 15

FINAL_TOP_K = 6

TARGET_WORDS = 180

OVERLAP_WORDS = 45


# ============================================================
# DEVICE
# ============================================================

if torch.cuda.is_available():

    DEVICE = "cuda"

    COMPUTE_TYPE = "float16"

else:

    DEVICE = "cpu"

    COMPUTE_TYPE = "int8"


# ============================================================
# SESSION STATE
# ============================================================

defaults = {

    "chunks": [],

    "transcript_segments": [],

    "faiss_index": None,

    "video_path": None,

    "video_name": None,

    "language": None,

    "duration": 0,

    "processed": False
}

for key, value in defaults.items():

    if key not in st.session_state:

        st.session_state[key] = value


# ============================================================
# MODEL LOADING
# ============================================================

@st.cache_resource
def load_whisper():

    return WhisperModel(

        WHISPER_MODEL,

        device=DEVICE,

        compute_type=COMPUTE_TYPE
    )


@st.cache_resource
def load_embedding_model():

    return SentenceTransformer(
        EMBEDDING_MODEL
    )


@st.cache_resource
def load_reranker():

    return CrossEncoder(
        RERANKER_MODEL
    )


# ============================================================
# TIMESTAMP
# ============================================================

def format_timestamp(seconds):

    seconds = int(
        max(0, seconds)
    )

    hours = seconds // 3600

    minutes = (
        seconds % 3600
    ) // 60

    secs = seconds % 60

    if hours > 0:

        return (
            f"{hours:02d}:"
            f"{minutes:02d}:"
            f"{secs:02d}"
        )

    return (
        f"{minutes:02d}:"
        f"{secs:02d}"
    )


# ============================================================
# VIDEO DURATION
# ============================================================

def get_video_duration(
    video_path
):

    command = [

        "ffprobe",

        "-v",
        "error",

        "-show_entries",
        "format=duration",

        "-of",
        "default=noprint_wrappers=1:nokey=1",

        str(video_path)
    ]

    result = subprocess.run(

        command,

        stdout=subprocess.PIPE,

        stderr=subprocess.PIPE,

        text=True
    )

    try:

        return float(
            result.stdout.strip()
        )

    except:

        return 0


# ============================================================
# AUDIO EXTRACTION
# ============================================================

def extract_audio(
    video_path
):

    audio_path = (

        AUDIO_DIR /
        f"{Path(video_path).stem}.wav"
    )

    command = [

        "ffmpeg",

        "-y",

        "-i",
        str(video_path),

        "-vn",

        "-ac",
        "1",

        "-ar",
        "16000",

        "-acodec",
        "pcm_s16le",

        str(audio_path)
    ]

    result = subprocess.run(

        command,

        stdout=subprocess.PIPE,

        stderr=subprocess.PIPE,

        text=True
    )

    if result.returncode != 0:

        raise RuntimeError(
            result.stderr[-3000:]
        )

    return audio_path


# ============================================================
# TRANSCRIPTION
# ============================================================

def transcribe_video(
    audio_path
):

    whisper = load_whisper()

    segments, info = (

        whisper.transcribe(

            str(audio_path),

            beam_size=5,

            vad_filter=True,

            word_timestamps=True,

            condition_on_previous_text=True
        )
    )

    results = []

    for segment in segments:

        text = (
            segment.text
            .strip()
        )

        if not text:
            continue

        results.append({

            "start":
                float(segment.start),

            "end":
                float(segment.end),

            "text":
                text
        })

    return (
        results,
        info.language
    )


# ============================================================
# SMART CHUNKING
# ============================================================

def create_chunks(

    segments,

    video_name,

    language
):

    results = []

    i = 0

    total = len(
        segments
    )

    while i < total:

        start_index = i

        word_count = 0

        # ----------------------------------------------------
        # Build chunk
        # ----------------------------------------------------

        while i < total:

            word_count += len(

                segments[i]["text"]
                .split()
            )

            i += 1

            if word_count >= TARGET_WORDS:

                break

        end_index = i

        selected = segments[
            start_index:end_index
        ]

        if not selected:

            continue

        text = " ".join(

            x["text"]

            for x in selected
        ).strip()

        start_time = (
            selected[0]["start"]
        )

        end_time = (
            selected[-1]["end"]
        )

        chunk_id = hashlib.sha1(

            (
                f"{video_name}"
                f"{start_time}"
                f"{end_time}"
            ).encode()

        ).hexdigest()[:12]

        results.append({

            "id":
                chunk_id,

            "video_name":
                video_name,

            "text":
                text,

            "start":
                start_time,

            "end":
                end_time,

            "start_timestamp":
                format_timestamp(
                    start_time
                ),

            "end_timestamp":
                format_timestamp(
                    end_time
                ),

            "language":
                language
        })

        # ----------------------------------------------------
        # OVERLAP
        # ----------------------------------------------------

        overlap_count = 0

        j = end_index - 1

        while (

            j >= start_index

            and

            overlap_count <
            OVERLAP_WORDS

        ):

            overlap_count += len(

                segments[j]["text"]
                .split()
            )

            j -= 1

        new_i = j + 1

        if new_i <= start_index:

            new_i = (
                start_index + 1
            )

        i = new_i

    return results


# ============================================================
# BUILD FAISS
# ============================================================

def build_faiss_index(
    lecture_chunks
):

    embedding_model = (
        load_embedding_model()
    )

    texts = [

        item["text"]

        for item in lecture_chunks
    ]

    embeddings = (
        embedding_model.encode(

            texts,

            batch_size=64,

            show_progress_bar=False,

            normalize_embeddings=True
        )
    )

    embeddings = np.asarray(

        embeddings,

        dtype="float32"
    )

    dimension = (
        embeddings.shape[1]
    )

    index = faiss.IndexFlatIP(
        dimension
    )

    index.add(
        embeddings
    )

    return index


# ============================================================
# RETRIEVE + RERANK
# ============================================================

def retrieve_chunks(
    question
):

    index = (
        st.session_state.faiss_index
    )

    lecture_chunks = (
        st.session_state.chunks
    )

    if (

        index is None

        or

        not lecture_chunks

    ):

        return []

    embedding_model = (
        load_embedding_model()
    )

    question_embedding = (

        embedding_model.encode(

            [question],

            normalize_embeddings=True
        )
    )

    question_embedding = np.asarray(

        question_embedding,

        dtype="float32"
    )

    k = min(

        INITIAL_TOP_K,

        len(lecture_chunks)
    )

    scores, indices = (

        index.search(

            question_embedding,

            k
        )
    )

    candidates = []

    for score, idx in zip(

        scores[0],

        indices[0]

    ):

        if idx < 0:

            continue

        item = dict(

            lecture_chunks[
                int(idx)
            ]
        )

        item[
            "retrieval_score"
        ] = float(score)

        candidates.append(
            item
        )

    # --------------------------------------------------------
    # RERANK
    # --------------------------------------------------------

    if len(candidates) > 1:

        reranker = (
            load_reranker()
        )

        pairs = [

            [
                question,
                item["text"]
            ]

            for item in candidates
        ]

        rerank_scores = (

            reranker.predict(
                pairs
            )
        )

        for item, score in zip(

            candidates,

            rerank_scores

        ):

            item[
                "rerank_score"
            ] = float(score)

        candidates.sort(

            key=lambda x:
                x["rerank_score"],

            reverse=True
        )

    return candidates[
        :FINAL_TOP_K
    ]


# ============================================================
# SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """

You are LectureMind, an expert AI teaching assistant.

Answer questions about recorded technical lectures.

STRICT RULES:

1. Use ONLY the supplied lecture evidence.

2. Never invent information.

3. Never invent timestamps.

4. Never claim something was taught unless supported
   by the supplied evidence.

5. If the answer cannot be found in the evidence,
say:

"I couldn't find that information in the indexed lecture."

6. Explain difficult concepts clearly.

7. Use intuition followed by technical explanation
when appropriate.

8. Cite evidence using [S1], [S2], [S3].

9. Mention timestamps when useful.

10. For comparisons, use a structured format.

"""


# ============================================================
# OPENROUTER
# ============================================================

def get_openrouter_client(
    api_key
):

    return OpenAI(

        api_key=api_key,

        base_url=
        "https://openrouter.ai/api/v1"
    )


# ============================================================
# GENERATE RAG ANSWER
# ============================================================

def generate_answer(

    question,

    retrieved_chunks,

    api_key,

    model

):

    client = get_openrouter_client(
        api_key
    )

    context_parts = []

    for i, item in enumerate(

        retrieved_chunks,

        start=1

    ):

        context_parts.append(

            f"""
[S{i}]

Lecture:
{item["video_name"]}

Timestamp:
{item["start_timestamp"]}
→
{item["end_timestamp"]}

Transcript:
{item["text"]}
"""
        )

    context = "\n".join(
        context_parts
    )

    prompt = f"""

STUDENT QUESTION:

{question}


LECTURE EVIDENCE:

{context}


TASK:

Answer the student's question.

Use ONLY the lecture evidence.

Cite supporting evidence with [S1], [S2], etc.

If the evidence is insufficient,
explicitly say so.

"""

    response = (

        client
        .chat
        .completions
        .create(

            model=model,

            messages=[

                {
                    "role":
                        "system",

                    "content":
                        SYSTEM_PROMPT
                },

                {
                    "role":
                        "user",

                    "content":
                        prompt
                }

            ],

            temperature=0.2,

            max_tokens=1500
        )
    )

    return (

        response
        .choices[0]
        .message
        .content
        .strip()
    )


# ============================================================
# STUDY GUIDE
# ============================================================

def generate_study_guide(

    api_key,

    model

):

    lecture_chunks = (
        st.session_state.chunks
    )

    if not lecture_chunks:

        return (
            "❌ Please process a lecture first."
        )

    if len(lecture_chunks) <= 15:

        selected = lecture_chunks

    else:

        positions = np.linspace(

            0,

            len(lecture_chunks) - 1,

            15

        ).astype(int)

        selected = [

            lecture_chunks[int(i)]

            for i in positions
        ]

    context = "\n\n".join(

        item["text"]

        for item in selected
    )

    client = get_openrouter_client(
        api_key
    )

    prompt = f"""

Create a high-quality study guide
from this lecture transcript.

Use ONLY the supplied transcript.

Include:

# Lecture Overview

# Key Concepts

# Important Definitions

# Important Formulas

# Examples Discussed

# Exam / Interview Important Points

# Common Mistakes

# 5 Quick Revision Questions

Do not invent information.

LECTURE:

{context}

"""

    response = (

        client
        .chat
        .completions
        .create(

            model=model,

            messages=[

                {
                    "role":
                        "system",

                    "content":
                        "You are an expert academic "
                        "teaching assistant. Use only "
                        "the supplied lecture."
                },

                {
                    "role":
                        "user",

                    "content":
                        prompt
                }

            ],

            temperature=0.2,

            max_tokens=2500
        )
    )

    return (

        response
        .choices[0]
        .message
        .content
        .strip()
    )


# ============================================================
# QUIZ
# ============================================================

def generate_quiz(

    api_key,

    model

):

    lecture_chunks = (
        st.session_state.chunks
    )

    if not lecture_chunks:

        return (
            "❌ Please process a lecture first."
        )

    if len(lecture_chunks) <= 10:

        selected = lecture_chunks

    else:

        positions = np.linspace(

            0,

            len(lecture_chunks) - 1,

            10

        ).astype(int)

        selected = [

            lecture_chunks[int(i)]

            for i in positions
        ]

    context = "\n\n".join(

        item["text"]

        for item in selected
    )

    client = get_openrouter_client(
        api_key
    )

    prompt = f"""

Create a 10-question quiz from this lecture.

Use ONLY the supplied transcript.

Create:

5 MCQs
3 conceptual questions
2 application questions

For every MCQ provide:

A.
B.
C.
D.

Then provide:

# Answer Key

Give the correct answer and
a one-sentence explanation.

LECTURE:

{context}

"""

    response = (

        client
        .chat
        .completions
        .create(

            model=model,

            messages=[

                {
                    "role":
                        "system",

                    "content":
                        "You are an expert technical "
                        "education assistant. Use only "
                        "the supplied lecture."
                },

                {
                    "role":
                        "user",

                    "content":
                        prompt
                }

            ],

            temperature=0.2,

            max_tokens=3000
        )
    )

    return (

        response
        .choices[0]
        .message
        .content
        .strip()
    )


# ============================================================
# HEADER
# ============================================================

st.markdown(
    '<div class="main-title">'
    '🎓 LectureMind'
    '</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'Timestamp-Aware RAG Teaching Assistant'
    '</div>',
    unsafe_allow_html=True
)

st.write(
    "Transform long technical lectures into "
    "an interactive AI tutor."
)

st.code(
    "Video → Whisper → Chunking → Embeddings → "
    "FAISS → Reranking → RAG → OpenRouter",
    language="text"
)


# ============================================================
# API KEY — MAIN SCREEN
# ============================================================

st.markdown(
    "## 🔐 OpenRouter Configuration"
)

col1, col2 = st.columns(
    [2, 1]
)

with col1:

    api_key = st.text_input(

        "OpenRouter API Key",

        type="password",

        placeholder="sk-or-v1-...",

        help=(
            "Your key is used only for "
            "OpenRouter LLM requests."
        )
    )

with col2:

    model = st.text_input(

        "OpenRouter Model",

        value="openai/gpt-5.2"
    )


if api_key:

    st.success(
        "✅ OpenRouter API key entered"
    )

else:

    st.info(
        "Enter your OpenRouter API key above "
        "to activate the AI Tutor, Study Guide "
        "and Quiz Generator."
    )


st.divider()


# ============================================================
# EXACTLY FOUR TABS
# ============================================================

tab1, tab2, tab3, tab4 = st.tabs(

    [
        "📚 Lecture",
        "🤖 AI Tutor",
        "📝 Study Guide",
        "🧪 Quiz Generator"
    ]
)


# ============================================================
# TAB 1 — LECTURE
# ============================================================

with tab1:

    st.header(
        "📚 Lecture Processing"
    )

    st.write(
        "Upload a lecture video and convert it "
        "into a searchable RAG knowledge base."
    )

    uploaded_file = st.file_uploader(

        "Upload Lecture Video",

        type=[
            "mp4",
            "mov",
            "mkv",
            "avi",
            "webm"
        ]
    )

    if uploaded_file:

        if st.button(

            "🚀 Process Lecture",

            type="primary",

            use_container_width=True
        ):

            try:

                with st.status(

                    "Processing lecture...",

                    expanded=True

                ) as status:

                    # ----------------------------------------
                    # SAVE VIDEO
                    # ----------------------------------------

                    video_path = (

                        VIDEO_DIR /
                        uploaded_file.name
                    )

                    with open(

                        video_path,

                        "wb"

                    ) as f:

                        f.write(
                            uploaded_file.getbuffer()
                        )

                    st.write(
                        "✅ Video uploaded"
                    )

                    # ----------------------------------------
                    # DURATION
                    # ----------------------------------------

                    duration = (
                        get_video_duration(
                            video_path
                        )
                    )

                    st.write(
                        "⏱️ Duration:",
                        format_timestamp(
                            duration
                        )
                    )

                    # ----------------------------------------
                    # AUDIO
                    # ----------------------------------------

                    st.write(
                        "🎵 Extracting audio..."
                    )

                    audio_path = (
                        extract_audio(
                            video_path
                        )
                    )

                    st.write(
                        "✅ Audio extracted"
                    )

                    # ----------------------------------------
                    # WHISPER
                    # ----------------------------------------

                    st.write(
                        "🗣️ Transcribing with "
                        "Faster-Whisper..."
                    )

                    segments, language = (
                        transcribe_video(
                            audio_path
                        )
                    )

                    st.write(
                        f"✅ {len(segments):,} "
                        "transcript segments"
                    )

                    # ----------------------------------------
                    # CHUNKING
                    # ----------------------------------------

                    st.write(
                        "🧩 Creating semantic chunks..."
                    )

                    lecture_chunks = create_chunks(

                        segments,

                        uploaded_file.name,

                        language
                    )

                    st.write(
                        f"✅ {len(lecture_chunks):,} "
                        "semantic chunks"
                    )

                    # ----------------------------------------
                    # EMBEDDINGS + FAISS
                    # ----------------------------------------

                    st.write(
                        "🧠 Creating embeddings..."
                    )

                    index = build_faiss_index(
                        lecture_chunks
                    )

                    st.write(
                        "✅ FAISS vector database created"
                    )

                    # ----------------------------------------
                    # SAVE STATE
                    # ----------------------------------------

                    st.session_state.chunks = (
                        lecture_chunks
                    )

                    st.session_state.transcript_segments = (
                        segments
                    )

                    st.session_state.faiss_index = (
                        index
                    )

                    st.session_state.video_path = (
                        str(video_path)
                    )

                    st.session_state.video_name = (
                        uploaded_file.name
                    )

                    st.session_state.language = (
                        language
                    )

                    st.session_state.duration = (
                        duration
                    )

                    st.session_state.processed = True

                    # ----------------------------------------
                    # SAVE DATA
                    # ----------------------------------------

                    with open(

                        DATA_DIR /
                        "transcript.json",

                        "w",

                        encoding="utf-8"

                    ) as f:

                        json.dump(

                            segments,

                            f,

                            ensure_ascii=False,

                            indent=2
                        )

                    with open(

                        DATA_DIR /
                        "chunks.json",

                        "w",

                        encoding="utf-8"

                    ) as f:

                        json.dump(

                            lecture_chunks,

                            f,

                            ensure_ascii=False,

                            indent=2
                        )

                    faiss.write_index(

                        index,

                        str(
                            DATA_DIR /
                            "lecturemind.faiss"
                        )
                    )

                    status.update(

                        label=(
                            "🎉 Lecture processed "
                            "successfully!"
                        ),

                        state="complete"
                    )

            except Exception as e:

                st.error(
                    f"❌ Error:\n\n{str(e)}"
                )

    # --------------------------------------------------------
    # CURRENT LECTURE
    # --------------------------------------------------------

    if st.session_state.processed:

        st.divider()

        st.subheader(
            "🎥 Current Lecture"
        )

        c1, c2, c3, c4 = st.columns(4)

        with c1:

            st.metric(

                "Duration",

                format_timestamp(
                    st.session_state.duration
                )
            )

        with c2:

            st.metric(

                "Segments",

                len(
                    st.session_state
                    .transcript_segments
                )
            )

        with c3:

            st.metric(

                "Chunks",

                len(
                    st.session_state.chunks
                )
            )

        with c4:

            st.metric(

                "Language",

                st.session_state.language
            )

        st.video(
            st.session_state.video_path
        )


# ============================================================
# TAB 2 — AI TUTOR
# ============================================================

with tab2:

    st.header(
        "🤖 AI Tutor"
    )

    st.write(
        "Ask questions about your indexed lecture."
    )

    if not st.session_state.processed:

        st.info(
            "📚 Process a lecture first."
        )

    elif not api_key:

        st.warning(
            "🔐 Enter your OpenRouter API key "
            "above to use the AI Tutor."
        )

    else:

        question = st.text_area(

            "Your Question",

            placeholder=(
                "Example: Explain cross-validation "
                "and why it is useful."
            ),

            height=120
        )

        ask_button = st.button(

            "🔍 Ask LectureMind",

            type="primary",

            use_container_width=True
        )

        if ask_button:

            if not question.strip():

                st.warning(
                    "Please enter a question."
                )

            else:

                with st.spinner(
                    "🔎 Searching lecture..."
                ):

                    results = retrieve_chunks(
                        question
                    )

                if not results:

                    st.warning(
                        "No relevant evidence found."
                    )

                else:

                    with st.spinner(
                        "🤖 Generating answer..."
                    ):

                        answer = generate_answer(

                            question,

                            results,

                            api_key,

                            model
                        )

                    st.subheader(
                        "💡 AI Answer"
                    )

                    st.markdown(
                        answer
                    )

                    st.divider()

                    st.subheader(
                        "📚 Retrieved Evidence"
                    )

                    for i, item in enumerate(

                        results,

                        start=1

                    ):

                        title = (

                            f"[S{i}] "
                            f"{item['start_timestamp']} → "
                            f"{item['end_timestamp']}"
                        )

                        with st.expander(
                            title,
                            expanded=(i == 1)
                        ):

                            st.markdown(
                                f"**Lecture:** "
                                f"{item['video_name']}"
                            )

                            st.markdown(
                                f"**Timestamp:** "
                                f"`{item['start_timestamp']}` "
                                f"→ "
                                f"`{item['end_timestamp']}`"
                            )

                            st.write(
                                item["text"]
                            )

                            c1, c2 = st.columns(2)

                            with c1:

                                st.metric(

                                    "Vector Score",

                                    f"{item.get('retrieval_score', 0):.4f}"
                                )

                            with c2:

                                st.metric(

                                    "Rerank Score",

                                    f"{item.get('rerank_score', 0):.4f}"
                                )


# ============================================================
# TAB 3 — STUDY GUIDE
# ============================================================

with tab3:

    st.header(
        "📝 AI Study Guide"
    )

    st.write(
        "Generate structured revision notes "
        "from your lecture."
    )

    if not st.session_state.processed:

        st.info(
            "📚 Process a lecture first."
        )

    elif not api_key:

        st.warning(
            "🔐 Enter your OpenRouter API key "
            "above to generate the study guide."
        )

    else:

        st.markdown(
            """
            The study guide includes:

            - 📌 Lecture overview
            - 🧠 Key concepts
            - 📖 Important definitions
            - 📐 Important formulas
            - 💡 Examples
            - 🎯 Exam/interview points
            - ⚠️ Common mistakes
            - ❓ Revision questions
            """
        )

        if st.button(

            "🧠 Generate Study Guide",

            type="primary",

            use_container_width=True
        ):

            with st.spinner(
                "🧠 Creating study guide..."
            ):

                guide = generate_study_guide(

                    api_key,

                    model
                )

            st.markdown(
                guide
            )


# ============================================================
# TAB 4 — QUIZ GENERATOR
# ============================================================

with tab4:

    st.header(
        "🧪 Quiz Generator"
    )

    st.write(
        "Generate a practice test based "
        "on the indexed lecture."
    )

    if not st.session_state.processed:

        st.info(
            "📚 Process a lecture first."
        )

    elif not api_key:

        st.warning(
            "🔐 Enter your OpenRouter API key "
            "above to generate the quiz."
        )

    else:

        st.markdown(
            """
            ### Quiz Structure

            **5 MCQs**

            **3 Conceptual Questions**

            **2 Application Questions**
            """
        )

        if st.button(

            "🎯 Generate 10 Questions",

            type="primary",

            use_container_width=True
        ):

            with st.spinner(
                "🧪 Creating quiz..."
            ):

                quiz = generate_quiz(

                    api_key,

                    model
                )

            st.markdown(
                quiz
            )


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "🎓 LectureMind • "
    "Timestamp-Aware RAG Teaching Assistant"
)
'''


# ================================================================
# 3. WRITE APP.PY
# ================================================================

with open(
    "/content/app.py",
    "w",
    encoding="utf-8"
) as f:

    f.write(app_code)

print("✅ Streamlit app created")


# ================================================================
# 4. STOP OLD STREAMLIT
# ================================================================

subprocess.run(
    "pkill -f 'streamlit run /content/app.py'",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)


# ================================================================
# 5. START STREAMLIT
# ================================================================

log_file = open(
    "/content/streamlit.log",
    "w"
)

process = subprocess.Popen(

    [
        "streamlit",
        "run",
        "/content/app.py",

        "--server.port",
        "8501",

        "--server.address",
        "0.0.0.0",

        "--server.headless",
        "true",

        "--browser.gatherUsageStats",
        "false"
    ],

    stdout=log_file,

    stderr=subprocess.STDOUT
)


# ================================================================
# 6. WAIT
# ================================================================

print("🚀 Starting Streamlit...")

time.sleep(8)


# ================================================================
# 7. OPEN COLAB STREAMLIT WINDOW
# ================================================================

try:

    from google.colab import output

    url = output.serve_kernel_port_as_window(
        8501
    )

    print("\n" + "=" * 70)
    print("🎓 LECTUREMIND IS READY")
    print("=" * 70)

    print(
        "\n🌐 Streamlit URL:"
    )

    print(url)

    print(
        "\nOpen the URL above."
    )

except Exception as e:

    print(
        "\n⚠️ Could not automatically "
        "create the Colab window."
    )

    print(
        "\nCheck Streamlit logs:"
    )

    print(
        open(
            "/content/streamlit.log",
            "r",
            errors="ignore"
        ).read()[-3000:]
    )

📦 Installing dependencies...
✅ Dependencies installed
✅ Streamlit app created
🚀 Starting Streamlit...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>


🎓 LECTUREMIND IS READY

🌐 Streamlit URL:
None

Open the URL above.


In [1]:
import subprocess
import time
import requests

print("1️⃣ Checking Streamlit process...")
print(
    subprocess.run(
        "ps aux | grep '[s]treamlit'",
        shell=True,
        capture_output=True,
        text=True
    ).stdout
)

print("\n2️⃣ Checking port 8501...")
print(
    subprocess.run(
        "curl -I http://127.0.0.1:8501",
        shell=True,
        capture_output=True,
        text=True
    ).stdout
)

print("\n3️⃣ Streamlit log:")
try:
    with open("/content/streamlit.log", "r") as f:
        print(f.read()[-5000:])
except Exception as e:
    print("No log file:", e)

1️⃣ Checking Streamlit process...


2️⃣ Checking port 8501...


3️⃣ Streamlit log:
No log file: [Errno 2] No such file or directory: '/content/streamlit.log'


In [2]:
# ================================================================
# 🎓 LECTUREMIND — FINAL COLAB + STREAMLIT VERSION
# ================================================================
#
# ONE CELL ONLY
#
# FIRST SCREEN:
#   🔐 OpenRouter API Key
#
# AFTER ACTIVATION:
#   📚 Lecture
#   🤖 AI Tutor
#   📝 Study Guide
#   🧪 Quiz Generator
#
# ================================================================

import os
import sys
import subprocess
import time
from pathlib import Path


# ================================================================
# 1. INSTALL PACKAGES
# ================================================================

print("=" * 70)
print("📦 INSTALLING LECTUREMIND")
print("=" * 70)

subprocess.run(
    "apt-get -qq update",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

subprocess.run(
    "apt-get -qq install -y ffmpeg",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

packages = [
    "streamlit",
    "openai",
    "faster-whisper",
    "sentence-transformers",
    "faiss-cpu",
    "requests"
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q"
    ] + packages,
    check=True
)

print("✅ Packages installed")


# ================================================================
# 2. CREATE APPLICATION
# ================================================================

APP_PATH = "/content/lecturemind_app.py"

app = r'''
# ================================================================
# LECTUREMIND STREAMLIT APPLICATION
# ================================================================

import streamlit as st

# ---------------------------------------------------------------
# IMPORTANT:
# Heavy ML libraries are NOT imported here.
# They are imported only when needed.
# This ensures the API-key screen loads first.
# ---------------------------------------------------------------

import os
import json
import hashlib
import subprocess
from pathlib import Path

import numpy as np


# ================================================================
# PAGE CONFIG
# ================================================================

st.set_page_config(
    page_title="LectureMind",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="collapsed"
)


# ================================================================
# CUSTOM CSS
# ================================================================

st.markdown(
    """
    <style>

    .title {
        font-size: 48px;
        font-weight: 800;
        margin-bottom: 0px;
    }

    .subtitle {
        font-size: 22px;
        font-weight: 600;
        color: #555;
    }

    .pipeline {
        font-family: monospace;
        font-size: 16px;
    }

    .api-box {
        padding: 20px;
        border-radius: 15px;
        border: 1px solid #ddd;
        background: #fafafa;
    }

    </style>
    """,
    unsafe_allow_html=True
)


# ================================================================
# DIRECTORIES
# ================================================================

BASE_DIR = Path("/content/lecturemind_data")

VIDEO_DIR = BASE_DIR / "videos"
AUDIO_DIR = BASE_DIR / "audio"
DATA_DIR = BASE_DIR / "data"

VIDEO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

AUDIO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# CONFIGURATION
# ================================================================

WHISPER_MODEL = "small"

EMBEDDING_MODEL = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

RERANKER_MODEL = (
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

TOP_K_RETRIEVAL = 15

TOP_K_FINAL = 6

CHUNK_WORDS = 180

OVERLAP_WORDS = 45


# ================================================================
# SESSION STATE
# ================================================================

if "activated" not in st.session_state:
    st.session_state.activated = False

if "api_key" not in st.session_state:
    st.session_state.api_key = ""

if "model_name" not in st.session_state:
    st.session_state.model_name = "openai/gpt-5.2"

if "chunks" not in st.session_state:
    st.session_state.chunks = []

if "segments" not in st.session_state:
    st.session_state.segments = []

if "faiss_index" not in st.session_state:
    st.session_state.faiss_index = None

if "video_path" not in st.session_state:
    st.session_state.video_path = None

if "video_name" not in st.session_state:
    st.session_state.video_name = None

if "duration" not in st.session_state:
    st.session_state.duration = 0

if "language" not in st.session_state:
    st.session_state.language = None


# ================================================================
# HEADER
# ================================================================

st.markdown(
    '<div class="title">🎓 LectureMind</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'Timestamp-Aware RAG Teaching Assistant'
    '</div>',
    unsafe_allow_html=True
)

st.write(
    "Transform long technical lectures into "
    "an interactive AI tutor."
)

st.code(
    "Video → Whisper → Chunking → Embeddings → "
    "FAISS → Reranking → RAG → OpenRouter",
    language="text"
)


# ================================================================
# API KEY SCREEN
# ================================================================

if not st.session_state.activated:

    st.markdown("## 🔐 Activate LectureMind")

    st.info(
        "Enter your OpenRouter API key to activate "
        "the AI teaching assistant."
    )

    st.markdown(
        '<div class="api-box">',
        unsafe_allow_html=True
    )

    api_key_input = st.text_input(
        "OpenRouter API Key",
        type="password",
        placeholder="sk-or-v1-...",
        help=(
            "Your API key is used for OpenRouter "
            "LLM requests."
        )
    )

    model_input = st.text_input(
        "OpenRouter Model",
        value=st.session_state.model_name,
        help=(
            "Enter a model available in your "
            "OpenRouter account."
        )
    )

    st.markdown(
        '</div>',
        unsafe_allow_html=True
    )

    st.write("")

    activate = st.button(
        "🚀 Activate LectureMind",
        type="primary",
        use_container_width=True
    )

    if activate:

        if not api_key_input.strip():

            st.error(
                "❌ Please enter your OpenRouter API key."
            )

        elif not api_key_input.startswith("sk-or-"):

            st.warning(
                "⚠️ The key does not look like a standard "
                "OpenRouter key. Please check it."
            )

        else:

            st.session_state.api_key = (
                api_key_input.strip()
            )

            st.session_state.model_name = (
                model_input.strip()
            )

            st.session_state.activated = True

            st.rerun()

    st.divider()

    st.markdown(
        """
        ### What happens after activation?

        **📚 Lecture**

        Upload a lecture → Whisper transcription →
        timestamps → semantic chunks → FAISS.

        **🤖 AI Tutor**

        Question → semantic retrieval → Cross-Encoder
        reranking → OpenRouter → grounded answer.

        **📝 Study Guide**

        Automatically generate structured revision notes.

        **🧪 Quiz Generator**

        Generate MCQs, conceptual questions and
        application questions from the lecture.

        """

    )

    st.stop()


# ================================================================
# AFTER API KEY ACTIVATION
# ================================================================

api_key = st.session_state.api_key

model_name = st.session_state.model_name


# ================================================================
# SIDEBAR
# ================================================================

with st.sidebar:

    st.title("🎓 LectureMind")

    st.success(
        "🔐 OpenRouter connected"
    )

    st.write(
        "**Model:**"
    )

    st.code(
        model_name
    )

    if st.button(
        "🔄 Change API Key"
    ):

        st.session_state.activated = False
        st.session_state.api_key = ""

        st.rerun()

    st.divider()

    st.write(
        "### Pipeline"
    )

    st.write(
        "🎥 Video"
    )

    st.write(
        "🗣️ Faster-Whisper"
    )

    st.write(
        "🧩 Chunking"
    )

    st.write(
        "🧠 Embeddings"
    )

    st.write(
        "🔎 FAISS"
    )

    st.write(
        "🎯 Reranking"
    )

    st.write(
        "🤖 OpenRouter"
    )


# ================================================================
# LAZY MODEL FUNCTIONS
# ================================================================

@st.cache_resource
def get_whisper():

    from faster_whisper import WhisperModel

    import torch

    if torch.cuda.is_available():

        return WhisperModel(
            WHISPER_MODEL,
            device="cuda",
            compute_type="float16"
        )

    return WhisperModel(
        WHISPER_MODEL,
        device="cpu",
        compute_type="int8"
    )


@st.cache_resource
def get_embedding_model():

    from sentence_transformers import SentenceTransformer

    return SentenceTransformer(
        EMBEDDING_MODEL
    )


@st.cache_resource
def get_reranker():

    from sentence_transformers import CrossEncoder

    return CrossEncoder(
        RERANKER_MODEL
    )


# ================================================================
# TIMESTAMP
# ================================================================

def timestamp(seconds):

    seconds = int(seconds)

    h = seconds // 3600

    m = (seconds % 3600) // 60

    s = seconds % 60

    if h > 0:

        return f"{h:02d}:{m:02d}:{s:02d}"

    return f"{m:02d}:{s:02d}"


# ================================================================
# VIDEO DURATION
# ================================================================

def video_duration(path):

    command = [
        "ffprobe",
        "-v",
        "error",
        "-show_entries",
        "format=duration",
        "-of",
        "default=noprint_wrappers=1:nokey=1",
        str(path)
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    try:
        return float(
            result.stdout.strip()
        )

    except:
        return 0


# ================================================================
# EXTRACT AUDIO
# ================================================================

def extract_audio(video):

    audio = (
        AUDIO_DIR /
        f"{Path(video).stem}.wav"
    )

    command = [
        "ffmpeg",
        "-y",
        "-i",
        str(video),
        "-vn",
        "-ac",
        "1",
        "-ar",
        "16000",
        "-acodec",
        "pcm_s16le",
        str(audio)
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:

        raise RuntimeError(
            result.stderr[-3000:]
        )

    return audio


# ================================================================
# TRANSCRIBE
# ================================================================

def transcribe(audio):

    whisper = get_whisper()

    segments, info = whisper.transcribe(

        str(audio),

        beam_size=5,

        vad_filter=True,

        word_timestamps=True,

        condition_on_previous_text=True
    )

    output = []

    for segment in segments:

        text = segment.text.strip()

        if text:

            output.append(
                {
                    "start":
                        float(segment.start),

                    "end":
                        float(segment.end),

                    "text":
                        text
                }
            )

    return output, info.language


# ================================================================
# CREATE CHUNKS
# ================================================================

def make_chunks(
    segments,
    filename,
    language
):

    chunks = []

    i = 0

    while i < len(segments):

        start_i = i

        words = 0

        while i < len(segments):

            words += len(
                segments[i]["text"].split()
            )

            i += 1

            if words >= CHUNK_WORDS:
                break

        selected = segments[
            start_i:i
        ]

        if not selected:
            continue

        text = " ".join(
            x["text"]
            for x in selected
        )

        start = selected[0]["start"]

        end = selected[-1]["end"]

        chunk_id = hashlib.sha1(
            f"{filename}_{start}_{end}".encode()
        ).hexdigest()[:12]

        chunks.append(
            {
                "id": chunk_id,

                "video_name": filename,

                "text": text,

                "start": start,

                "end": end,

                "start_timestamp":
                    timestamp(start),

                "end_timestamp":
                    timestamp(end),

                "language": language
            }
        )

        # overlap
        overlap = 0

        j = i - 1

        while (
            j >= start_i
            and overlap < OVERLAP_WORDS
        ):

            overlap += len(
                segments[j]["text"].split()
            )

            j -= 1

        next_i = j + 1

        if next_i <= start_i:

            next_i = start_i + 1

        i = next_i

    return chunks


# ================================================================
# CREATE FAISS
# ================================================================

def create_faiss(chunks):

    import faiss

    model = get_embedding_model()

    texts = [
        x["text"]
        for x in chunks
    ]

    embeddings = model.encode(

        texts,

        batch_size=64,

        show_progress_bar=False,

        normalize_embeddings=True
    )

    embeddings = np.asarray(
        embeddings,
        dtype="float32"
    )

    index = faiss.IndexFlatIP(
        embeddings.shape[1]
    )

    index.add(
        embeddings
    )

    return index


# ================================================================
# RETRIEVE + RERANK
# ================================================================

def retrieve(question):

    import faiss

    index = (
        st.session_state.faiss_index
    )

    chunks = (
        st.session_state.chunks
    )

    if index is None:

        return []

    model = get_embedding_model()

    query_embedding = model.encode(
        [question],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    k = min(
        TOP_K_RETRIEVAL,
        len(chunks)
    )

    scores, indices = index.search(
        query_embedding,
        k
    )

    candidates = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx < 0:
            continue

        item = dict(
            chunks[int(idx)]
        )

        item["retrieval_score"] = float(
            score
        )

        candidates.append(item)

    # Cross Encoder
    if len(candidates) > 1:

        reranker = get_reranker()

        pairs = [
            [
                question,
                item["text"]
            ]
            for item in candidates
        ]

        scores = reranker.predict(
            pairs
        )

        for item, score in zip(
            candidates,
            scores
        ):

            item["rerank_score"] = float(
                score
            )

        candidates.sort(
            key=lambda x:
                x["rerank_score"],
            reverse=True
        )

    return candidates[
        :TOP_K_FINAL
    ]


# ================================================================
# OPENROUTER CLIENT
# ================================================================

def openrouter():

    from openai import OpenAI

    return OpenAI(
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1"
    )


# ================================================================
# RAG ANSWER
# ================================================================

def answer_question(
    question,
    results
):

    context = ""

    for i, item in enumerate(
        results,
        start=1
    ):

        context += f"""

[S{i}]

Lecture:
{item["video_name"]}

Timestamp:
{item["start_timestamp"]}
→
{item["end_timestamp"]}

Transcript:
{item["text"]}

"""

    system = """

You are LectureMind, an expert AI teaching assistant.

Answer using ONLY the supplied lecture evidence.

Rules:

1. Never invent information.
2. Never invent timestamps.
3. Cite evidence using [S1], [S2], etc.
4. Explain technical concepts clearly.
5. If evidence is insufficient, say:
   "I couldn't find that information in the indexed lecture."
6. Keep answers educational and reasonably concise.

"""

    prompt = f"""

STUDENT QUESTION:

{question}


LECTURE EVIDENCE:

{context}


Answer the question using only the evidence.

"""

    client = openrouter()

    response = (
        client.chat.completions.create(

            model=model_name,

            messages=[
                {
                    "role":
                        "system",
                    "content":
                        system
                },
                {
                    "role":
                        "user",
                    "content":
                        prompt
                }
            ],

            temperature=0.2,

            max_tokens=1600
        )
    )

    return (
        response
        .choices[0]
        .message
        .content
        .strip()
    )


# ================================================================
# STUDY GUIDE
# ================================================================

def study_guide():

    chunks = (
        st.session_state.chunks
    )

    if len(chunks) <= 15:

        selected = chunks

    else:

        positions = np.linspace(
            0,
            len(chunks) - 1,
            15
        ).astype(int)

        selected = [
            chunks[int(i)]
            for i in positions
        ]

    transcript = "\n\n".join(
        x["text"]
        for x in selected
    )

    prompt = f"""

Create a comprehensive but concise study guide
from the lecture transcript below.

Use ONLY the transcript.

Include:

# Lecture Overview

# Key Concepts

# Important Definitions

# Important Formulas

# Examples

# Exam Important Points

# Common Mistakes

# Quick Revision Questions

Do not invent information.

LECTURE:

{transcript}

"""

    response = (
        openrouter()
        .chat
        .completions
        .create(

            model=model_name,

            messages=[
                {
                    "role":
                        "system",
                    "content":
                        "You are an expert academic tutor."
                },
                {
                    "role":
                        "user",
                    "content":
                        prompt
                }
            ],

            temperature=0.2,

            max_tokens=3000
        )
    )

    return (
        response
        .choices[0]
        .message
        .content
        .strip()
    )


# ================================================================
# QUIZ
# ================================================================

def create_quiz():

    chunks = (
        st.session_state.chunks
    )

    if len(chunks) <= 10:

        selected = chunks

    else:

        positions = np.linspace(
            0,
            len(chunks) - 1,
            10
        ).astype(int)

        selected = [
            chunks[int(i)]
            for i in positions
        ]

    transcript = "\n\n".join(
        x["text"]
        for x in selected
    )

    prompt = f"""

Create a 10-question quiz using ONLY
the lecture transcript.

Structure:

5 MCQs
3 conceptual questions
2 application questions

For every MCQ provide A, B, C and D.

Then provide:

# Answer Key

For every answer give a short explanation.

Do not invent information.

LECTURE:

{transcript}

"""

    response = (
        openrouter()
        .chat
        .completions
        .create(

            model=model_name,

            messages=[
                {
                    "role":
                        "system",
                    "content":
                        "You are an expert technical educator."
                },
                {
                    "role":
                        "user",
                    "content":
                        prompt
                }
            ],

            temperature=0.2,

            max_tokens=3500
        )
    )

    return (
        response
        .choices[0]
        .message
        .content
        .strip()
    )


# ================================================================
# FOUR TABS ONLY
# ================================================================

tab1, tab2, tab3, tab4 = st.tabs(
    [
        "📚 Lecture",
        "🤖 AI Tutor",
        "📝 Study Guide",
        "🧪 Quiz Generator"
    ]
)


# ================================================================
# TAB 1 — LECTURE
# ================================================================

with tab1:

    st.header(
        "📚 Lecture Processing"
    )

    st.write(
        "Upload a lecture video and build "
        "your timestamp-aware RAG knowledge base."
    )

    uploaded = st.file_uploader(

        "Upload Lecture Video",

        type=[
            "mp4",
            "mov",
            "mkv",
            "avi",
            "webm"
        ]
    )

    if uploaded:

        if st.button(
            "🚀 Process Lecture",
            type="primary",
            use_container_width=True
        ):

            try:

                with st.status(
                    "Processing lecture...",
                    expanded=True
                ) as status:

                    # Save video
                    video_path = (
                        VIDEO_DIR /
                        uploaded.name
                    )

                    with open(
                        video_path,
                        "wb"
                    ) as f:

                        f.write(
                            uploaded.getbuffer()
                        )

                    st.write(
                        "✅ Video uploaded"
                    )

                    # Duration
                    duration = video_duration(
                        video_path
                    )

                    st.write(
                        f"⏱️ Duration: "
                        f"{timestamp(duration)}"
                    )

                    # Audio
                    st.write(
                        "🎵 Extracting audio..."
                    )

                    audio = extract_audio(
                        video_path
                    )

                    st.write(
                        "✅ Audio extracted"
                    )

                    # Whisper
                    st.write(
                        "🗣️ Loading Faster-Whisper..."
                    )

                    segments, language = (
                        transcribe(audio)
                    )

                    st.write(
                        f"✅ Transcribed "
                        f"{len(segments):,} segments"
                    )

                    # Chunking
                    st.write(
                        "🧩 Creating chunks..."
                    )

                    chunks = make_chunks(
                        segments,
                        uploaded.name,
                        language
                    )

                    st.write(
                        f"✅ Created "
                        f"{len(chunks):,} chunks"
                    )

                    # Embeddings
                    st.write(
                        "🧠 Creating embeddings..."
                    )

                    index = create_faiss(
                        chunks
                    )

                    st.write(
                        "✅ FAISS index created"
                    )

                    # Save state
                    st.session_state.segments = (
                        segments
                    )

                    st.session_state.chunks = (
                        chunks
                    )

                    st.session_state.faiss_index = (
                        index
                    )

                    st.session_state.video_path = (
                        str(video_path)
                    )

                    st.session_state.video_name = (
                        uploaded.name
                    )

                    st.session_state.duration = (
                        duration
                    )

                    st.session_state.language = (
                        language
                    )

                    # Save JSON
                    with open(
                        DATA_DIR / "transcript.json",
                        "w",
                        encoding="utf-8"
                    ) as f:

                        json.dump(
                            segments,
                            f,
                            ensure_ascii=False,
                            indent=2
                        )

                    with open(
                        DATA_DIR / "chunks.json",
                        "w",
                        encoding="utf-8"
                    ) as f:

                        json.dump(
                            chunks,
                            f,
                            ensure_ascii=False,
                            indent=2
                        )

                    status.update(
                        label=(
                            "🎉 Lecture processed successfully!"
                        ),
                        state="complete"
                    )

            except Exception as e:

                st.error(
                    "❌ Lecture processing failed"
                )

                st.exception(e)

    # Current lecture
    if st.session_state.video_path:

        st.divider()

        st.subheader(
            "🎥 Current Lecture"
        )

        c1, c2, c3, c4 = st.columns(4)

        c1.metric(
            "Duration",
            timestamp(
                st.session_state.duration
            )
        )

        c2.metric(
            "Segments",
            len(
                st.session_state.segments
            )
        )

        c3.metric(
            "Chunks",
            len(
                st.session_state.chunks
            )
        )

        c4.metric(
            "Language",
            st.session_state.language
        )

        st.video(
            st.session_state.video_path
        )


# ================================================================
# TAB 2 — AI TUTOR
# ================================================================

with tab2:

    st.header(
        "🤖 AI Tutor"
    )

    if not st.session_state.chunks:

        st.info(
            "📚 Process a lecture first."
        )

    else:

        question = st.text_area(

            "Ask a question about the lecture",

            placeholder=(
                "Example: Explain cross-validation "
                "and why it is useful."
            ),

            height=120
        )

        ask = st.button(
            "🔍 Ask LectureMind",
            type="primary",
            use_container_width=True
        )

        if ask:

            if not question.strip():

                st.warning(
                    "Please enter a question."
                )

            else:

                try:

                    with st.spinner(
                        "🔎 Retrieving evidence..."
                    ):

                        results = retrieve(
                            question
                        )

                    if not results:

                        st.warning(
                            "No relevant evidence found."
                        )

                    else:

                        with st.spinner(
                            "🤖 Generating grounded answer..."
                        ):

                            answer = (
                                answer_question(
                                    question,
                                    results
                                )
                            )

                        st.subheader(
                            "💡 Answer"
                        )

                        st.markdown(
                            answer
                        )

                        st.divider()

                        st.subheader(
                            "📚 Evidence"
                        )

                        for i, item in enumerate(
                            results,
                            start=1
                        ):

                            with st.expander(
                                f"[S{i}] "
                                f"{item['start_timestamp']} → "
                                f"{item['end_timestamp']}",
                                expanded=(i == 1)
                            ):

                                st.write(
                                    item["text"]
                                )

                                c1, c2 = st.columns(2)

                                c1.metric(
                                    "Retrieval Score",
                                    f"{item.get('retrieval_score', 0):.4f}"
                                )

                                c2.metric(
                                    "Rerank Score",
                                    f"{item.get('rerank_score', 0):.4f}"
                                )

                except Exception as e:

                    st.error(
                        "❌ OpenRouter request failed"
                    )

                    st.exception(e)


# ================================================================
# TAB 3 — STUDY GUIDE
# ================================================================

with tab3:

    st.header(
        "📝 AI Study Guide"
    )

    if not st.session_state.chunks:

        st.info(
            "📚 Process a lecture first."
        )

    else:

        st.write(
            "Generate structured revision notes "
            "from the indexed lecture."
        )

        st.markdown(
            """
            The study guide includes:

            - 📌 Lecture overview
            - 🧠 Key concepts
            - 📖 Important definitions
            - 📐 Important formulas
            - 💡 Examples
            - 🎯 Exam-important points
            - ⚠️ Common mistakes
            - ❓ Revision questions
            """
        )

        if st.button(
            "🧠 Generate Study Guide",
            type="primary",
            use_container_width=True
        ):

            try:

                with st.spinner(
                    "🧠 Generating study guide..."
                ):

                    guide = study_guide()

                st.markdown(
                    guide
                )

            except Exception as e:

                st.error(
                    "❌ Study guide generation failed"
                )

                st.exception(e)


# ================================================================
# TAB 4 — QUIZ
# ================================================================

with tab4:

    st.header(
        "🧪 Quiz Generator"
    )

    if not st.session_state.chunks:

        st.info(
            "📚 Process a lecture first."
        )

    else:

        st.write(
            "Generate a lecture-based practice test."
        )

        st.markdown(
            """
            ### Quiz Structure

            **5 MCQs**

            **3 Conceptual Questions**

            **2 Application Questions**
            """
        )

        if st.button(
            "🎯 Generate 10 Questions",
            type="primary",
            use_container_width=True
        ):

            try:

                with st.spinner(
                    "🧪 Generating quiz..."
                ):

                    quiz = create_quiz()

                st.markdown(
                    quiz
                )

            except Exception as e:

                st.error(
                    "❌ Quiz generation failed"
                )

                st.exception(e)


# ================================================================
# FOOTER
# ================================================================

st.divider()

st.caption(
    "🎓 LectureMind | "
    "Timestamp-Aware RAG Teaching Assistant"
)
'''


# ================================================================
# 3. WRITE FILE
# ================================================================

with open(
    APP_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(app)

print("✅ Application written:")
print(APP_PATH)


# ================================================================
# 4. STOP OLD APP
# ================================================================

print("\n🛑 Stopping old Streamlit processes...")

subprocess.run(
    "pkill -f 'streamlit run /content/lecturemind_app.py'",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)


# ================================================================
# 5. START STREAMLIT
# ================================================================

LOG_PATH = "/content/lecturemind_streamlit.log"

log = open(
    LOG_PATH,
    "w",
    buffering=1
)

print("🚀 Starting Streamlit...")

process = subprocess.Popen(

    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        APP_PATH,

        "--server.port",
        "8501",

        "--server.address",
        "0.0.0.0",

        "--server.headless",
        "true",

        "--server.enableCORS",
        "false",

        "--server.enableXsrfProtection",
        "false",

        "--browser.gatherUsageStats",
        "false"
    ],

    stdout=log,

    stderr=subprocess.STDOUT,

    start_new_session=True
)


# ================================================================
# 6. WAIT FOR STREAMLIT
# ================================================================

print("⏳ Waiting for Streamlit...")

import requests

ready = False

for i in range(30):

    time.sleep(1)

    try:

        r = requests.get(
            "http://127.0.0.1:8501",
            timeout=2
        )

        if r.status_code in [200, 303]:

            ready = True

            print(
                f"✅ Streamlit started "
                f"after {i+1} seconds"
            )

            break

    except Exception:
        pass


# ================================================================
# 7. SHOW ERROR IF NOT STARTED
# ================================================================

if not ready:

    print("\n❌ Streamlit failed to start.")

    print("\n========== STREAMLIT LOG ==========\n")

    if os.path.exists(LOG_PATH):

        with open(
            LOG_PATH,
            "r",
            errors="ignore"
        ) as f:

            print(
                f.read()[-10000:]
            )

    raise RuntimeError(
        "Streamlit did not start. "
        "See the error above."
    )


# ================================================================
# 8. OPEN COLAB PORT
# ================================================================

print("\n" + "=" * 70)
print("🎓 LECTUREMIND IS READY")
print("=" * 70)

print(
    "\n🔐 FIRST SCREEN WILL ASK FOR:"
)

print(
    "   → OpenRouter API Key"
)

print(
    "\nThen you will see exactly:"
)

print(
    "   1. 📚 Lecture"
)

print(
    "   2. 🤖 AI Tutor"
)

print(
    "   3. 📝 Study Guide"
)

print(
    "   4. 🧪 Quiz Generator"
)

try:

    from google.colab import output

    print(
        "\n🌐 Opening Streamlit..."
    )

    url = output.serve_kernel_port_as_window(
        8501
    )

    print(
        "\nStreamlit URL:"
    )

    print(url)

except Exception as e:

    print(
        "\n⚠️ Could not automatically open "
        "the Streamlit window."
    )

    print(
        "Error:",
        e
    )

    print(
        "\nYou can check the server at "
        "port 8501 in Colab."
    )

print(
    "\n✅ Server is running."
)

📦 INSTALLING LECTUREMIND
✅ Packages installed
✅ Application written:
/content/lecturemind_app.py

🛑 Stopping old Streamlit processes...
🚀 Starting Streamlit...
⏳ Waiting for Streamlit...
✅ Streamlit started after 3 seconds

🎓 LECTUREMIND IS READY

🔐 FIRST SCREEN WILL ASK FOR:
   → OpenRouter API Key

Then you will see exactly:
   1. 📚 Lecture
   2. 🤖 AI Tutor
   3. 📝 Study Guide
   4. 🧪 Quiz Generator

🌐 Opening Streamlit...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>


Streamlit URL:
None

✅ Server is running.
